In [ ]:
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def add_label(label, data):
    return {x: f"{x}{label}" for x in data}

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory (where figure will be saved): {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "green_algae_mit",
    "green_algae_plt",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
]

polarity_3bin = {
    "++": "same",
    "--": "same",
    "+-": "conv",
    "-+": "div",
}

group_titles = {
    "fungi_mit": "Fungi (mitochondria)",
    "metazoans_mit": "Metazoans (mitochondria)",
    "plants_mit": "Plants (mitochondria)",
    "protists_mit": "Protists (mitochondria)",
    "green_algae_mit": "Green algae (mitochondria)",
    "plants_plt": "Plants (plastids)",
    "protists_plt": "Protists (plastids)",
    "green_algae_plt": "Green algae (plastids)",
}

label_3bin = set(polarity_3bin.values())

COLOR_SAME = "#D55E00"  # vermillion
COLOR_CONV = "#56B4E9"  # sky blue
COLOR_DIV = "#03045E"  # deep navy

In [ ]:
group_rows = []

for g in groups:
    print(group_titles[g])
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t")
    igs = pl.read_csv(BASE / g / "summary_igs_intergenic.tsv", separator="\t")

    igs = igs.join(
        tsv.select(["AN", "Genome_length"]),
        on="AN",
        how="left",
        validate="m:1",
    )

    igs = igs.with_columns(
        pl.col("Polarity").replace_strict(polarity_3bin).alias("polarity_3bin")
    )

    med = igs.group_by(["AN", "polarity_3bin"]).agg(
        pl.col("Length").count().alias("count"),
        pl.col("Length").sum().alias("total_bp"),
    )

    wide_counts = med.pivot(values="count", index="AN", on="polarity_3bin")
    wide_counts = wide_counts.rename(add_label("_count", label_3bin))

    wide_totals = med.pivot(values="total_bp", index="AN", on="polarity_3bin")
    wide_totals = wide_totals.rename(add_label("_total", label_3bin))

    igs_sizes = igs.group_by("AN").agg(pl.col("Length").sum().alias("total_igs_size"))

    wide = wide_counts.join(wide_totals, on="AN", how="left", validate="1:1").join(
        igs_sizes, on="AN", how="left", validate="m:1"
    )

    # --- enrichment ratios ---
    sum_total = wide["total_igs_size"].sum()
    sum_counts = (
        wide["conv_count"].sum() + wide["div_count"].sum() + wide["same_count"].sum()
    )

    P_obs_conv = wide["conv_total"].sum() / sum_total
    P_obs_div = wide["div_total"].sum() / sum_total
    P_obs_same = wide["same_total"].sum() / sum_total

    P_exp_conv = wide["conv_count"].sum() / sum_counts
    P_exp_div = wide["div_count"].sum() / sum_counts
    P_exp_same = wide["same_count"].sum() / sum_counts

    E_conv = P_obs_conv / P_exp_conv
    E_div = P_obs_div / P_exp_div
    E_same = P_obs_same / P_exp_same

    group_rows.append({"group": g, "E_conv": E_conv, "E_div": E_div, "E_same": E_same})

    print(
        f"Observed pct (by length): Convergent: {P_obs_conv:.4f}, Divergent: {P_obs_div:.4f}, Same: {P_obs_same:.4f}"
    )
    print(
        f"Expected pct (by count):  Convergent: {P_exp_conv:.4f}, Divergent: {P_exp_div:.4f}, Same: {P_exp_same:.4f}"
    )
    print(
        f"Enrichment Ratios:        Convergent: {E_conv:.4f}, Divergent: {E_div:.4f}, Same: {E_same:.4f}"
    )
    print()

data_3bin = pl.DataFrame(group_rows)

In [ ]:
x = list(range(len(data_3bin)))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))

bars_conv = ax.bar(
    [i - width for i in x],
    data_3bin["E_conv"],
    width,
    label="Convergent",
    color=COLOR_CONV,
    zorder=3,
)
bars_div = ax.bar(
    x, data_3bin["E_div"], width, label="Divergent", color=COLOR_DIV, zorder=3
)
bars_same = ax.bar(
    [i + width for i in x],
    data_3bin["E_same"],
    width,
    label="Same",
    color=COLOR_SAME,
    zorder=3,
)

ax.axhline(
    1, color="black", linewidth=1.2, linestyle="--", zorder=2, label="E = 1 (null)"
)

ax.set_xticks(x)
ax.set_xticklabels(
    [group_titles[g].replace(" (", "\n(") for g in data_3bin["group"]],
    fontsize=10,
)
ax.set_ylabel("Enrichment ratio (E)", fontsize=11)
ax.set_ylim(
    0, max([*data_3bin["E_conv"], *data_3bin["E_div"], *data_3bin["E_same"]]) * 1.15
)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

legend_elements = [
    mpatches.Patch(color=COLOR_CONV, label="Convergent"),
    mpatches.Patch(color=COLOR_DIV, label="Divergent"),
    mpatches.Patch(color=COLOR_SAME, label="Same"),
    plt.Line2D(
        [0], [0], color="black", linestyle="--", linewidth=1.2, label="E = 1 (null)"
    ),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_title(
    "Enrichment ratios of convergent, divergent, and same polarity IGRs across groups",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)

plt.tight_layout()
plt.savefig("figure3.png", dpi=300, bbox_inches="tight")
plt.show()